# safepyrun — Safe Python Sandbox for LLM Tools

**safepyrun** is an allowlist-based Python sandbox that lets LLMs execute code safely in your real environment. Instead of isolating code in a container (which cuts it off from libraries, data, and tools it needs), safepyrun runs in-process with controlled access to a curated subset of Python's stdlib.

## How It Works

```
LLM wants to run code
       |
       v
 RunPython(code)
       |
       v
 RestrictedPython compiles to modified AST
       |
       v
 Every attribute/item access goes through gatekeepers
       |
       v
 Allowlist checks: is this callable permitted?
       |
       v
 Result returned (stdout, stderr, return value)
```

## Key Features
- **Allowlist-based**: Only permitted callables are accessible
- **In-process**: Access to your real objects and data
- **Async-native**: Supports `await`, `async for`, `async with`
- **Write policies**: Controlled filesystem writes via `ok_dests`
- **State persistence**: Variables/functions ending with `_` persist across calls

## Basic Usage

In [ ]:
from safepyrun import RunPython

# Create a sandbox instance
pyrun = RunPython()

In [ ]:
# Simple expression
await pyrun('1 + 1')

In [ ]:
# Print output is captured
await pyrun('print("Hello from the sandbox!"); 42')

## Module Access

The default allowlist covers a large subset of the standard library:

In [ ]:
# Math operations
await pyrun('import math; math.sqrt(144)')

In [ ]:
# String processing with regex
await pyrun('''
import re
text = "Hello World 123 Foo 456"
numbers = re.findall(r"\\d+", text)
f"Found numbers: {numbers}"
''')

In [ ]:
# JSON parsing
await pyrun('''
import json
data = json.loads('{"name": "dialeng", "version": "0.1.0"}')
f"Project: {data['name']} v{data['version']}"
''')

In [ ]:
# Path operations (read-only by default)
await pyrun('''
from pathlib import Path
p = Path(".")
files = sorted([f.name for f in p.iterdir() if f.is_file()])[:5]
f"First 5 files: {files}"
''')

## The `_` Suffix Convention — Persistence & Callability

This is safepyrun's core security mechanism. It controls two things:

1. **Persistence**: Only variables/functions ending with `_` survive between `pyrun` calls. Everything else is discarded after the call completes.
2. **Callability**: Only `_`-suffixed functions (or those registered via `allow()`) are permitted to be called. Non-`_` callables are blocked.

**Why?** Without this, an LLM could define arbitrary functions and call them across sandbox invocations, effectively bypassing the allowlist. The `_` suffix is an explicit opt-in.

### Internal flow:
```
pyrun("def greet_(name): ...")
  → code executes in restricted env
  → greet_ defined in locals (loc)
  → _export() checks: name ends with _? YES
  → copies greet_ to persistent globals (g)

Next call: pyrun("greet_('Joe')")
  → tools dict built from g → includes greet_
  → _callable_ok("greet_") → True (ends with _)
  → greet_ is available and callable ✓
```

### What happens WITHOUT the `_` suffix:
```
pyrun("def greet(name): ...")
  → greet defined in locals (loc)
  → _export() checks: name ends with _? NO
  → greet is NOT copied to globals
  → discarded after call

Next call: pyrun("greet('Joe')")
  → greet not in globals → NameError
  → "greet is not available in this sandbox"
```

In [ ]:
# ✅ Persistent variable (ends with _)
await pyrun("x_ = 42")
print("Set x_ = 42")

In [ ]:
# x_ persists across calls
await pyrun("x_ * 2")

In [ ]:
# ✅ Persistent function (ends with _)
await pyrun('def greet_(name): return f"hello {name}"')
print("Defined greet_")

In [ ]:
# greet_ persists and is callable across calls
await pyrun("greet_('Joe')")

In [ ]:
# ❌ Non-persistent variable (no _ suffix) — discarded after the call
await pyrun("y = 99")
print("Set y = 99 (no _ suffix)")

In [ ]:
# y was not exported — this will fail with NameError
await pyrun("y")

In [ ]:
# ❌ Non-persistent function (no _ suffix) — also fails across calls
await pyrun('def hello(x): return f"hello {x}"')
print("Defined hello (no _ suffix)")

# This will fail: hello was not exported
result = await pyrun("hello('Joe')")
print(f"Result: {result}")

## Limitation: No Recursive Functions *Defined Inside pyrun*

Functions defined inside `pyrun` **cannot call themselves recursively** — even with the `_` suffix. This is not a safepyrun design choice; it's a fundamental consequence of how Python's `exec()` works with separate globals and locals dicts.

### Why it happens

When `pyrun` executes code, it uses `eval(compiled_code, rg, loc)` where:
- `rg` = restricted globals dict (contains builtins, allowed tools, previously exported `_` vars)
- `loc` = empty locals dict (where new definitions go)

When you define `def fib_(n): ... fib_(n-1)`:
1. `fib_` is created and stored in `loc` (locals)
2. Python sets `fib_.__globals__ = rg` (the globals dict)
3. When `fib_` tries to call itself, Python looks up `"fib_"` in `fib_.__globals__` (= `rg`)
4. But `fib_` is in `loc`, **not** in `rg` → **NameError**

This happens even with the `_` suffix, even within the same `pyrun` call, and even across calls (the function retains the old `rg` from when it was defined). This is **not documented** in safepyrun's docs — it's an inherent Python `exec()` behavior.

### Workarounds

1. **Use iterative implementations** instead of recursion (inside `pyrun`)
2. **Define recursive functions in normal Python** and register them via `allow()` — their `__globals__` is the real module globals dict, which contains themselves

In [ ]:
# ❌ Recursive function — fails even with _ suffix
await pyrun("def fib_(n): return n if n <= 1 else fib_(n-1) + fib_(n-2)")
result = await pyrun("fib_(5)")
print(f"Recursive fib_(5): {result}")  # Will show NameError for fib_

In [ ]:
# ❌ Even defining + calling in the SAME pyrun call fails for recursion
result = await pyrun("""
def fib(n): return n if n <= 1 else fib(n-1) + fib(n-2)
fib(5)
""")
print(f"Same-call recursive fib(5): {result}")  # Also fails

In [ ]:
# ✅ Workaround: use iterative implementations instead
await pyrun("""
def fib_(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a
""")
await pyrun("[fib_(i) for i in range(10)]")

In [ ]:
# ✅ Workaround: define recursive functions in normal Python + allow() them
from safepyrun import allow

def fib_recursive(n): return n if n <= 1 else fib_recursive(n-1) + fib_recursive(n-2)
allow('fib_recursive')

# This works because fib_recursive's __globals__ is the real module globals
# (which contains fib_recursive itself), not the sandbox's restricted globals
await pyrun('fib_recursive(10)')

## What Gets Blocked

Dangerous operations like filesystem writes, process spawning, and system modification are blocked:

In [ ]:
# Attempting to delete a file — blocked
try:
    await pyrun('import os; os.remove("/tmp/test")')
except Exception as e:
    print(f"Blocked: {type(e).__name__}: {e}")

In [ ]:
# Attempting to spawn a subprocess — blocked
try:
    await pyrun('import subprocess; subprocess.run(["ls"])')
except Exception as e:
    print(f"Blocked: {type(e).__name__}: {e}")

## Extending the Allowlist with `allow()`

Register your own functions or third-party library methods:

In [ ]:
from safepyrun import allow

# Define and allow a custom function
def greet(name): return f"Hello, {name}!"
allow('greet')

await pyrun('greet("World")')

## Write Permissions with `ok_dests`

By default, all filesystem writes are blocked. Use `ok_dests` to allow controlled writing:

In [ ]:
# Create a sandbox with write access to /tmp
pyrun_w = RunPython(ok_dests=['/tmp'])

# This works — writing to an allowed destination
await pyrun_w("Path('/tmp/safepyrun_test.txt').write_text('hello from sandbox')")

In [ ]:
# Verify the write worked
await pyrun_w("Path('/tmp/safepyrun_test.txt').read_text()")

In [ ]:
# But writing outside allowed destinations is still blocked
try:
    await pyrun_w("Path('/etc/evil.txt').write_text('bad')")
except PermissionError as e:
    print(f"Blocked: {e}")

## Async Support

The sandbox is async-native — `await`, `async for`, and `async with` all work:

In [ ]:
await pyrun('''
import asyncio
async def compute(n): return n * 10
results = await asyncio.gather(compute(1), compute(2), compute(3))
f"Async results: {results}"
''')

## Integration with Dialeng

In Dialeng, `pyrun` is registered as a **built-in LLM tool**. When the AI needs to execute Python code safely during a prompt response, it can call `pyrun` automatically — no `&` prefix needed.

The sandbox instance is created with `ok_dests=['.']` so the AI can write files relative to the current working directory.

Try creating a prompt cell and asking the AI to "use pyrun to calculate the first 20 prime numbers" to see it in action!

## Cleanup

In [ ]:
# Clean up test file
from pathlib import Path
Path('/tmp/safepyrun_test.txt').unlink(missing_ok=True)
print("Cleaned up test files")